[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davinan/COPAL/blob/main/applications/RALPH_blastp_inference.ipynb)

# RALPH BLASTP inference

User-facing entry point for the RALPH BLAST baseline. Given a LuxR-homolog
query sequence, it BLASTs against the RALPH reference database (10,496 RALPH
homologs + 153 RefAHL benchmark receptors) and shows, for each top hit, the
predicted ligand rankings under **COPAL (ensemble)** or any individual model
(**AF3, RF3, Boltz2, Boltz2Affinity**) selected from a dropdown — with that
model's score, the hit's mpKT, and, for RefAHL receptors, the
experimentally-known cognate ligand.

## Running on Google Colab

Click the **Open in Colab** badge above (or open the notebook from the GitHub
repo). Colab launches a fresh runtime in your own Google account, then hit
**Runtime → Run all**: the first (**Setup**) cell clones the repo, installs the
package + BLAST+, and builds the database automatically — no local setup needed.

## Running locally

1. From the repo root: `uv sync` (installs `ralph` + `copal` + deps).
2. Install BLAST+ (`blastp`, `makeblastdb` on PATH) — see `INSTALLATION.md`.
3. Launch Jupyter from `applications/`
   (`cd applications && jupyter notebook RALPH_blastp_inference.ipynb`), or set
   the `RALPH_DATABASE` environment variable to the database directory.

The **Setup** cell decompresses + builds the BLAST index on first run
(one-time); re-runs are no-ops.

## Using it

Edit `query_sequence` in the next cell, then **Run all**.

In [ ]:
#@title Change the variables below, then hit `Runtime` -> `Run all`
query_sequence = 'MQHWLDKLTDLAAIQGDECILKDGLADLAEHFGFTGYAYLHIQHKHTIAVTNYHRDWRSAYFENNFDKLDPVVKRAKSRKHVFAWSGEQERSRLSKEERAFYAHAADFGIRSGITIPIKTANGSMSMFTLASERPAIDLDREIDAAAAAGAVGQLHARISFLQTTPTVEDAAWLDPKEATYLRWIAVGMTMEEVADVEGVKYNSVRVKLREAMKRFDVRSKAHLTALAIRRKLI' #@param {type:"string"}
#@markdown  Change the variable below to change how many sequence hits to use from BLASTP
top_k_BLASTP_hits = 50 #@param {type:"raw"}
top_k = top_k_BLASTP_hits

In [ ]:
#@title Setup — installs deps + builds the database (run once; Colab + local)
import os, sys, subprocess, shutil

def _sh(cmd):
    subprocess.run(cmd, shell=isinstance(cmd, str), check=True)

# On Google Colab: clone the repo, install the package, install BLAST+.
if 'google.colab' in sys.modules:
    REPO_URL = "https://github.com/davinan/COPAL"   # public repo
    REPO_DIR = "/content/COPAL"
    if not os.path.isdir(REPO_DIR):
        _sh(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])
    os.chdir(REPO_DIR)
    _sh([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
    _sh("apt-get -qq update && apt-get -qq install -y ncbi-blast+")
    os.environ["RALPH_DATABASE"] = os.path.join(
        REPO_DIR, "baselines", "ralph_blast", "ralph_database")

# Resolve the database directory: RALPH_DATABASE env override, else the in-repo
# default (works when Jupyter is launched from applications/).
RALPH_DB = os.environ.get(
    "RALPH_DATABASE",
    os.path.abspath(os.path.join(os.getcwd(), "..", "baselines", "ralph_blast", "ralph_database")),
)
os.environ["RALPH_DATABASE"] = RALPH_DB

# Materialize the DB (decompress .xz + makeblastdb) if the BLAST index is absent.
setup_sh = os.path.join(os.path.dirname(RALPH_DB.rstrip("/")), "setup_db.sh")
if not os.path.exists(os.path.join(RALPH_DB, "ralph.pin")) and os.path.exists(setup_sh):
    print("Building the RALPH BLAST database (one-time)...")
    _sh(["bash", setup_sh])

if shutil.which("blastp") is None:
    raise RuntimeError("blastp not on PATH. Install BLAST+ (see INSTALLATION.md).")
print("RALPH database ready:", RALPH_DB)

In [ ]:
#@title Load the ralph package + database
import os
import sys

import pandas as pd
import numpy as np

# ralph ships in-tree under src/ralph/. Add it to sys.path so `import ralph`
# works even without an editable install (uv sync / pip install -e .).
sys.path.append("./src")
from ralph import (
    blastp_query,
    load_scores,
    load_model_scores,
    load_ligands,
    interactive_blastp_results,
)

# RALPH_DATABASE was resolved + built in the Setup cell above.
database_path = os.environ["RALPH_DATABASE"]

df = load_scores(database_path=database_path)                     # wide rankings + mpKT
model_scores_df = load_model_scores(database_path=database_path)  # per-pair model scores
ligand_df = load_ligands(database_path=database_path)             # ligand structures

In [ ]:
#@title Running BLASTP query on RALPH and Displaying Aggregated Rankings
# blastp_query is imported from the ralph package in the load cell above.

# database_path is already defined in the setup cell above.
db_path = os.path.join(database_path, "ralph")

hits = blastp_query(
    query_seq=query_sequence,
    db_path=db_path,
    max_hits=top_k,
)

hits = hits.sort_values("bitscore", ascending=False).head(top_k)

scores = df[df.luxr_name.isin(hits.sseqid)]

In [ ]:
#@title Interactive results — ranked ligands per hit + model
# Two dropdowns drive the view:
#  - "Select Hit"  picks the BLASTP hit
#  - "Model"       picks which model's ligand ranking to show (COPAL ensemble by
#                  default, or AF3, RF3, Boltz2, Boltz2Affinity)
# For the selected hit + model you get:
#  - the query-vs-hit sequence alignment
#  - the top-K ligands for that model, each annotated with the model's score
#  - the hit's mpKT (and, for RefAHL benchmark receptors, the known cognate ligand)
interactive_blastp_results(
    hits_df=hits,
    scores_df=df,
    query_sequence=query_sequence,
    ligand_df=ligand_df,
    model_scores_df=model_scores_df,
)